# CNN Assignment Solution

Rename this notebook to `YOUR_BITS_ID_cnn_assignment.ipynb` before submission. Fill in the student details below so the notebook metadata matches your actual submission.

## Student Information
- BITS ID: REPLACE_ME
- Name: REPLACE_ME
- Email: REPLACE_ME
- Date: REPLACE_ME

## Approach
- Dataset: Cats vs Dogs balanced subset downloaded from the Microsoft Kaggle archive
- Framework: PyTorch
- Custom model: 3-block CNN with MaxPooling and Global Average Pooling
- Transfer learning: frozen ResNet18 feature extractor with GAP and a custom classifier head
- Split: 85/15 train-test with a validation split carved from the training data

In [ ]:
# If you are running in a fresh Colab or VM, install the required packages first.
# !pip install torch torchvision tqdm seaborn scikit-learn pandas matplotlib pillow certifi

In [2]:
from __future__ import annotations

import copy
import json
import platform
import random
import subprocess
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
)
print(f'Using device: {device}')

student_info = {
    'BITS ID': 'REPLACE_ME',
    'Name': 'REPLACE_ME',
    'Email': 'REPLACE_ME',
    'Date': 'REPLACE_ME',
}
student_info

Using device: mps


{'BITS ID': 'REPLACE_ME',
 'Name': 'REPLACE_ME',
 'Email': 'REPLACE_ME',
 'Date': 'REPLACE_ME'}

In [ ]:
DATA_ROOT = Path('assignment_data')
DATA_ROOT.mkdir(exist_ok=True)

CATS_DOGS_URL = 'https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip'
RESNET18_WEIGHTS_URL = 'https://download.pytorch.org/models/resnet18-f37072fd.pth'

RAW_ZIP_PATH = DATA_ROOT / 'kagglecatsanddogs_5340.zip'
EXTRACT_DIR = DATA_ROOT / 'cats_vs_dogs_raw'
PET_IMAGES_DIR = EXTRACT_DIR / 'PetImages'
RESNET18_WEIGHTS_PATH = DATA_ROOT / 'resnet18-f37072fd.pth'

def download_file(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination
    print(f'Downloading {url} -> {destination}')
    subprocess.run(['curl', '-L', url, '-o', str(destination)], check=True)
    return destination

def ensure_cats_vs_dogs_dataset() -> Path:
    if not PET_IMAGES_DIR.exists():
        download_file(CATS_DOGS_URL, RAW_ZIP_PATH)
        with zipfile.ZipFile(RAW_ZIP_PATH, 'r') as archive:
            archive.extractall(EXTRACT_DIR)
    return PET_IMAGES_DIR

def ensure_resnet18_weights() -> Path:
    return download_file(RESNET18_WEIGHTS_URL, RESNET18_WEIGHTS_PATH)

dataset_dir = ensure_cats_vs_dogs_dataset()
weights_path = ensure_resnet18_weights()
print(f'Dataset directory: {dataset_dir}')
print(f'ResNet18 weights: {weights_path}')

In [ ]:
def collect_valid_image_paths(root: Path, limit_per_class: int = 1200) -> pd.DataFrame:
    rows = []
    label_map = {'Cat': 'cat', 'Dog': 'dog'}
    for folder_name, label in label_map.items():
        folder = root / folder_name
        valid_count = 0
        for path in sorted(folder.glob('*.jpg')):
            try:
                with Image.open(path) as image:
                    image.verify()
            except Exception:
                continue
            rows.append({'path': str(path), 'label': label})
            valid_count += 1
            if valid_count >= limit_per_class:
                break
    dataframe = pd.DataFrame(rows)
    dataframe['label_idx'] = dataframe['label'].map({'cat': 0, 'dog': 1})
    return dataframe.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

dataset_df = collect_valid_image_paths(dataset_dir, limit_per_class=1200)
class_counts = dataset_df['label'].value_counts().sort_index()

dataset_name = 'Cats vs Dogs (balanced subset)'
dataset_source = 'Microsoft Kaggle Cats and Dogs archive'
n_samples = int(len(dataset_df))
n_classes = int(dataset_df['label'].nunique())
samples_per_class = f"min: {class_counts.min()}, max: {class_counts.max()}, avg: {class_counts.mean():.0f}"
image_shape = [160, 160, 3]
problem_type = 'classification'
primary_metric = 'accuracy'
metric_justification = (
    'The subset is balanced across the two classes, so accuracy is an appropriate primary metric. '
    'Precision, recall, and F1-score are also reported to confirm balanced performance.'
)

print('\n' + '=' * 70)
print('DATASET INFORMATION')
print(f'Dataset: {dataset_name}')
print(f'Source: {dataset_source}')
print(f'Total Samples: {n_samples}')
print(f'Number of Classes: {n_classes}')
print(f'Samples per Class: {samples_per_class}')
print(f'Image Shape: {image_shape}')
print(f'Primary Metric: {primary_metric}')
print(f'Metric Justification: {metric_justification}')

class_counts.to_frame(name='count')

In [ ]:
def estimate_rgb_statistics(paths: list[str], sample_size: int = 96) -> tuple[np.ndarray, np.ndarray]:
    sample_paths = paths[:sample_size]
    pixels = []
    for path in sample_paths:
        with Image.open(path).convert('RGB') as image:
            resized = image.resize((160, 160))
            array = np.asarray(resized, dtype=np.float32) / 255.0
            pixels.append(array.reshape(-1, 3))
    stacked = np.concatenate(pixels, axis=0)
    return stacked.mean(axis=0), stacked.std(axis=0)

rgb_mean, rgb_std = estimate_rgb_statistics(dataset_df['path'].tolist())
print(f'Approximate RGB mean: {rgb_mean.round(4)}')
print(f'Approximate RGB std:  {rgb_std.round(4)}')

sample_rows = dataset_df.groupby('label', group_keys=False).head(4)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for axis, (_, row) in zip(axes.flatten(), sample_rows.iterrows()):
    with Image.open(row['path']).convert('RGB') as image:
        axis.imshow(image.resize((160, 160)))
    axis.set_title(row['label'].title())
    axis.axis('off')
plt.suptitle('Sample Images from Each Class', fontsize=14)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index.str.title(), y=class_counts.values, palette='Set2')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Images')
plt.show()

In [ ]:
train_df, test_df = train_test_split(
    dataset_df,
    test_size=0.15,
    random_state=SEED,
    stratify=dataset_df['label_idx'],
)
train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df['label_idx'],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_test_ratio = '85/15'
train_samples = int(len(train_df) + len(val_df))
test_samples = int(len(test_df))

print(f'Train/Test Split: {train_test_ratio}')
print(f'Training Samples: {train_samples}')
print(f'Validation Samples: {len(val_df)}')
print(f'Test Samples: {test_samples}')

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class ImageFrameDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        with Image.open(row['path']).convert('RGB') as image:
            image = image.copy()
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label_idx'])

train_dataset = ImageFrameDataset(train_df, transform=train_transform)
val_dataset = ImageFrameDataset(val_df, transform=eval_transform)
test_dataset = ImageFrameDataset(test_df, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

class_names = ['cat', 'dog']
print(f'Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}, Test batches: {len(test_loader)}')

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, n_classes),
        )
        self.conv_layers = 3
        self.pooling_layers = 3

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.features(inputs)
        pooled = self.gap(features)
        return self.classifier(pooled)

def build_custom_cnn(input_shape, n_classes):
    return CustomCNN(n_classes).to(device)

class TransferResNet18(nn.Module):
    def __init__(self, feature_extractor: nn.Module, n_classes: int):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(256, n_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.feature_extractor(inputs)
        pooled = self.gap(features)
        return self.classifier(pooled)

pretrained_model_name = 'ResNet18'

def build_transfer_learning_model(base_model_name, input_shape, n_classes):
    if base_model_name != 'ResNet18':
        raise ValueError('This notebook is configured for ResNet18.')
    backbone = models.resnet18(weights=None)
    state_dict = torch.load(weights_path, map_location='cpu')
    backbone.load_state_dict(state_dict)
    feature_extractor = nn.Sequential(*list(backbone.children())[:-2])
    for parameter in feature_extractor.parameters():
        parameter.requires_grad = False
    model = TransferResNet18(feature_extractor, n_classes).to(device)
    return model

def count_parameters(model: nn.Module) -> tuple[int, int]:
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable

def count_parameterized_layers(model: nn.Module, trainable: bool | None = None) -> int:
    count = 0
    for module in model.modules():
        params = list(module.parameters(recurse=False))
        if not params:
            continue
        is_trainable = any(parameter.requires_grad for parameter in params)
        if trainable is None or is_trainable == trainable:
            count += 1
    return count

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_predictions = []
    all_targets = []

    progress = tqdm(loader, leave=False)
    for images, targets in progress:
        images = images.to(device)
        targets = targets.to(device)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        predictions = torch.argmax(logits, dim=1)
        all_predictions.extend(predictions.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    avg_accuracy = accuracy_score(all_targets, all_predictions)
    return avg_loss, avg_accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}
    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_accuracy = run_epoch(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_accuracy'].append(train_accuracy)
        history['val_accuracy'].append(val_accuracy)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(
            f'Epoch {epoch:02d}/{epochs} | '
            f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
            f'train_acc={train_accuracy:.4f} | val_acc={val_accuracy:.4f}'
        )

    model.load_state_dict(best_state)
    return history

def evaluate_model(model, loader, class_names):
    model.eval()
    predictions = []
    targets = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            batch_predictions = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(batch_predictions)
            targets.extend(labels.numpy())

    metrics = {
        'accuracy': accuracy_score(targets, predictions),
        'precision': precision_score(targets, predictions, average='macro', zero_division=0),
        'recall': recall_score(targets, predictions, average='macro', zero_division=0),
        'f1_score': f1_score(targets, predictions, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(targets, predictions),
        'classification_report': classification_report(targets, predictions, target_names=class_names, zero_division=0),
        'targets': np.asarray(targets),
        'predictions': np.asarray(predictions),
    }
    return metrics

def plot_history(history, title):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(epochs, history['val_loss'], label='Val Loss', marker='o')
    axes[0].set_title(f'{title} Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(epochs, history['train_accuracy'], label='Train Accuracy', marker='o')
    axes[1].plot(epochs, history['val_accuracy'], label='Val Accuracy', marker='o')
    axes[1].set_title(f'{title} Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

def plot_confusion(cm, labels, title):
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

def show_sample_predictions(model, dataframe, transform, title, count=6):
    sample_df = dataframe.sample(n=min(count, len(dataframe)), random_state=SEED).reset_index(drop=True)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()

    model.eval()
    with torch.no_grad():
        for axis, (_, row) in zip(axes, sample_df.iterrows()):
            with Image.open(row['path']).convert('RGB') as image:
                display_image = image.resize((160, 160))
                input_tensor = transform(display_image).unsqueeze(0).to(device)
            prediction = int(torch.argmax(model(input_tensor), dim=1).item())
            axis.imshow(display_image)
            axis.set_title(f"True: {row['label']} | Pred: {class_names[prediction]}")
            axis.axis('off')

    for axis in axes[len(sample_df):]:
        axis.axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
print('\n' + '=' * 70)
print('CUSTOM CNN TRAINING')

custom_cnn = build_custom_cnn(image_shape, n_classes)
custom_cnn_total_parameters, custom_cnn_trainable_parameters = count_parameters(custom_cnn)
print(custom_cnn)
print(f'Total parameters: {custom_cnn_total_parameters:,}')

custom_learning_rate = 0.001
custom_epochs = 5
custom_batch_size = 32
custom_optimizer_name = 'Adam'
custom_loss_name = 'cross_entropy'

custom_criterion = nn.CrossEntropyLoss()
custom_optimizer = torch.optim.Adam(custom_cnn.parameters(), lr=custom_learning_rate)

custom_cnn_start_time = time.time()
custom_history = train_model(custom_cnn, train_loader, val_loader, custom_criterion, custom_optimizer, custom_epochs)
custom_cnn_training_time = time.time() - custom_cnn_start_time

custom_cnn_initial_loss = float(custom_history['train_loss'][0])
custom_cnn_final_loss = float(custom_history['train_loss'][-1])

print(f'Training completed in {custom_cnn_training_time:.2f} seconds')
print(f'Initial Loss: {custom_cnn_initial_loss:.4f}')
print(f'Final Loss: {custom_cnn_final_loss:.4f}')

print('\nCUSTOM CNN EVALUATION')
custom_metrics = evaluate_model(custom_cnn, test_loader, class_names)
custom_cnn_accuracy = float(custom_metrics['accuracy'])
custom_cnn_precision = float(custom_metrics['precision'])
custom_cnn_recall = float(custom_metrics['recall'])
custom_cnn_f1 = float(custom_metrics['f1_score'])

print('\nCustom CNN Performance:')
print(f'Accuracy:  {custom_cnn_accuracy:.4f}')
print(f'Precision: {custom_cnn_precision:.4f}')
print(f'Recall:    {custom_cnn_recall:.4f}')
print(f'F1-Score:  {custom_cnn_f1:.4f}')
print(custom_metrics['classification_report'])

plot_history(custom_history, 'Custom CNN')
plot_confusion(custom_metrics['confusion_matrix'], class_names, 'Custom CNN Confusion Matrix')
show_sample_predictions(custom_cnn, test_df, eval_transform, 'Custom CNN Sample Predictions')

In [ ]:
print('\n' + '=' * 70)
print('TRANSFER LEARNING IMPLEMENTATION')

transfer_model = build_transfer_learning_model(pretrained_model_name, image_shape, n_classes)
frozen_layers = count_parameterized_layers(transfer_model, trainable=False)
trainable_layers = count_parameterized_layers(transfer_model, trainable=True)
total_parameters, trainable_parameters = count_parameters(transfer_model)

print(f'Base Model: {pretrained_model_name}')
print(f'Frozen Layers: {frozen_layers}')
print(f'Trainable Layers: {trainable_layers}')
print(f'Total Parameters: {total_parameters:,}')
print(f'Trainable Parameters: {trainable_parameters:,}')
print('Using Global Average Pooling: YES')

tl_learning_rate = 0.0003
tl_epochs = 3
tl_batch_size = 32
tl_optimizer = 'Adam'
tl_loss_name = 'cross_entropy'

transfer_criterion = nn.CrossEntropyLoss()
transfer_optimizer = torch.optim.Adam(filter(lambda parameter: parameter.requires_grad, transfer_model.parameters()), lr=tl_learning_rate)

tl_start_time = time.time()
tl_history = train_model(transfer_model, train_loader, val_loader, transfer_criterion, transfer_optimizer, tl_epochs)
tl_training_time = time.time() - tl_start_time

tl_initial_loss = float(tl_history['train_loss'][0])
tl_final_loss = float(tl_history['train_loss'][-1])

print(f'Training completed in {tl_training_time:.2f} seconds')
print(f'Initial Loss: {tl_initial_loss:.4f}')
print(f'Final Loss: {tl_final_loss:.4f}')

print('\nTRANSFER LEARNING EVALUATION')
tl_metrics = evaluate_model(transfer_model, test_loader, class_names)
tl_accuracy = float(tl_metrics['accuracy'])
tl_precision = float(tl_metrics['precision'])
tl_recall = float(tl_metrics['recall'])
tl_f1 = float(tl_metrics['f1_score'])

print('\nTransfer Learning Performance:')
print(f'Accuracy:  {tl_accuracy:.4f}')
print(f'Precision: {tl_precision:.4f}')
print(f'Recall:    {tl_recall:.4f}')
print(f'F1-Score:  {tl_f1:.4f}')
print(tl_metrics['classification_report'])

plot_history(tl_history, 'Transfer Learning')
plot_confusion(tl_metrics['confusion_matrix'], class_names, 'Transfer Learning Confusion Matrix')
show_sample_predictions(transfer_model, test_df, eval_transform, 'Transfer Learning Sample Predictions')

In [ ]:
print('\n' + '=' * 70)
print('MODEL COMPARISON')

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Training Time (s)', 'Parameters'],
    'Custom CNN': [
        custom_cnn_accuracy,
        custom_cnn_precision,
        custom_cnn_recall,
        custom_cnn_f1,
        custom_cnn_training_time,
        custom_cnn_total_parameters,
    ],
    'Transfer Learning': [
        tl_accuracy,
        tl_precision,
        tl_recall,
        tl_f1,
        tl_training_time,
        total_parameters,
    ],
})
print(comparison_df.to_string(index=False))

metrics_only = comparison_df.iloc[:4].copy()
metrics_long = metrics_only.melt(id_vars='Metric', var_name='Model', value_name='Score')
plt.figure(figsize=(9, 5))
sns.barplot(data=metrics_long, x='Metric', y='Score', hue='Model', palette='Set1')
plt.ylim(0, 1.05)
plt.title('Metric Comparison')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(custom_metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Custom CNN')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(tl_metrics['confusion_matrix'], annot=True, fmt='d', cmap='Greens', ax=axes[1], xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('Transfer Learning')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

def generate_analysis_text() -> str:
    winner = 'Transfer learning' if tl_accuracy >= custom_cnn_accuracy else 'The custom CNN'
    accuracy_gap = abs(tl_accuracy - custom_cnn_accuracy)
    faster_model = 'transfer learning' if tl_training_time < custom_cnn_training_time else 'the custom CNN'
    return (
        f"{winner} performed better on the balanced Cats vs Dogs subset, with an accuracy gap of {accuracy_gap:.3f}. "
        f"The pre-trained ResNet18 converged faster because its frozen backbone already contained reusable visual features, while the custom CNN had to learn edges, textures, and shapes from scratch. "
        f"Global Average Pooling kept both models compact at the classifier end, reducing the chance of overfitting compared with a large Flatten-plus-Dense block. "
        f"The transfer model had far more total parameters ({total_parameters:,}) but only {trainable_parameters:,} remained trainable, which lowered optimization cost even though inference stayed heavier. "
        f"Overall, {faster_model} trained faster in this run. Transfer learning is the better default when data is moderate and time is limited, while a custom CNN is useful when model size, architecture control, or domain-specific experimentation matters most."
    )

analysis_text = generate_analysis_text()
analysis_word_count = len(analysis_text.split())

print('\n' + '=' * 70)
print('ANALYSIS')
print(analysis_text)
print(f'Analysis word count: {analysis_word_count} words')
if analysis_word_count > 200:
    print('Warning: Analysis exceeds 200 words (guideline)')
else:
    print('Analysis within word count guideline')

In [ ]:
def get_assignment_results():
    framework_used = 'pytorch'

    results = {
        'dataset_name': dataset_name,
        'dataset_source': dataset_source,
        'n_samples': n_samples,
        'n_classes': n_classes,
        'samples_per_class': samples_per_class,
        'image_shape': image_shape,
        'problem_type': problem_type,
        'primary_metric': primary_metric,
        'metric_justification': metric_justification,
        'train_samples': train_samples,
        'test_samples': test_samples,
        'train_test_ratio': train_test_ratio,
        'custom_cnn': {
            'framework': framework_used,
            'architecture': {
                'conv_layers': custom_cnn.conv_layers,
                'pooling_layers': custom_cnn.pooling_layers,
                'has_global_average_pooling': True,
                'output_layer': 'softmax',
                'total_parameters': custom_cnn_total_parameters,
            },
            'training_config': {
                'learning_rate': custom_learning_rate,
                'n_epochs': custom_epochs,
                'batch_size': custom_batch_size,
                'optimizer': custom_optimizer_name,
                'loss_function': custom_loss_name,
            },
            'initial_loss': custom_cnn_initial_loss,
            'final_loss': custom_cnn_final_loss,
            'training_time_seconds': custom_cnn_training_time,
            'accuracy': custom_cnn_accuracy,
            'precision': custom_cnn_precision,
            'recall': custom_cnn_recall,
            'f1_score': custom_cnn_f1,
        },
        'transfer_learning': {
            'framework': framework_used,
            'base_model': pretrained_model_name,
            'frozen_layers': frozen_layers,
            'trainable_layers': trainable_layers,
            'has_global_average_pooling': True,
            'total_parameters': total_parameters,
            'trainable_parameters': trainable_parameters,
            'training_config': {
                'learning_rate': tl_learning_rate,
                'n_epochs': tl_epochs,
                'batch_size': tl_batch_size,
                'optimizer': tl_optimizer,
                'loss_function': tl_loss_name,
            },
            'initial_loss': tl_initial_loss,
            'final_loss': tl_final_loss,
            'training_time_seconds': tl_training_time,
            'accuracy': tl_accuracy,
            'precision': tl_precision,
            'recall': tl_recall,
            'f1_score': tl_f1,
        },
        'analysis': analysis_text,
        'analysis_word_count': analysis_word_count,
        'custom_cnn_loss_decreased': custom_cnn_final_loss < custom_cnn_initial_loss,
        'transfer_learning_loss_decreased': tl_final_loss < tl_initial_loss,
    }
    return results

assignment_results = get_assignment_results()
print('\n' + '=' * 70)
print('ASSIGNMENT RESULTS SUMMARY')
print(json.dumps(assignment_results, indent=2))

In [ ]:
print('ENVIRONMENT INFORMATION')
print(f'Python version: {platform.python_version()}')
print(f'PyTorch version: {torch.__version__}')
print(f'Platform: {platform.platform()}')
print('\nREQUIRED: Add a screenshot of your Google Colab or BITS Virtual Lab account details below this cell before submission.')

## Final Checklist
- Replace the student information placeholders.
- Rename the file to match your real BITS ID.
- Run Kernel -> Restart & Run All so every output is visible.
- Insert your environment screenshot in a markdown cell after the environment section.
- Verify the JSON summary prints without errors before submission.